### DLT Pipeline

In [0]:
import dlt
import pyspark.sql.functions as F


### Streaming Table 

In [0]:
# Expectations 
rules = {
    "valid_id": "id IS NOT NULL",
    "valid_user_id": "user_id IS NOT NULL",
    "valid_created_at": "created_at IS NOT NULL",
}

In [0]:

@dlt.view(name="FactTripPost_stage")
@dlt.expect_all_or_drop(rules)
def FactTripPost_stage():
    df = (
        spark.readStream.table("travel_journal_catalog.silver.trip_posts")
        .filter(F.col("flag") == False)
        .withColumn(
            "post_date_id",
            F.date_format(F.col("created_at"), "yyyyMMdd").cast("int"),
        )
    )
    return df

## Fact Table

### Step 2: Fact Table - fact_trip_post

In [0]:
@dlt.table(
    name = "fact_trip_post",
    table_properties={"quality": "gold"}
)

def fact_trip_post():
  """ Joins staging posts stream with dim_account to validate keys and builds Gold Fact"""

  posts_df    = dlt.read_stream("FactTripPost_stage")
  accounts_df = dlt.read("dim_user")

  fact_trip_post_df = posts_df.alias("posts").join(
        accounts_df.alias("accounts"),
        (F.col("posts.user_id") == F.col("accounts.user_id"))          # fixed: user_id
        & (F.col("posts.created_at") >= F.col("accounts.__START_AT"))
        & (
            (F.col("posts.created_at") < F.col("accounts.__END_AT"))    # fixed: __END_AT
            | F.col("accounts.__END_AT").isNull()                       # grouped with the line above
        ),
        how="inner",
    ).select(                                                           # chained directly, no stray ')'
        F.col("posts.id").alias("trip_post_id"),
        F.col("accounts.DimUserKey").alias("user_key"),
        F.col("posts.user_id"),
        F.col("posts.post_date_id"),                                    # listed once, with comma
        F.col("posts.trip_rating"),
        F.col("posts.trip_name"),
        F.col("posts.total_distance"),
        F.col("posts.caption"),
        F.col("posts.created_at").alias("post_created_at"),
    )

  return fact_trip_post_df


#### STEP 3: Bridge Tables for Multi-Tags

In [0]:
@dlt.table(
    name="bridge_trip_post_country",
)

def bridge_trip_post_country():
    bridge_trip_post_country_df = spark.readStream.table("travel_journal_catalog.silver.location_country_tag").filter(F.col("flag") == False).select(
        F.col("trip_post_id"),
        F.col("country_code"))
    
    return bridge_trip_post_country_df

In [0]:
@dlt.table(
    name="bridge_trip_post_activity"
)

def bridge_trip_post_activity():
    bridge_trip_post_activity_df = spark.readStream.table("travel_journal_catalog.silver.activity_tag").filter(F.col("flag") == False).select(
        F.col("trip_post_id"),
        F.col("activity_code")
        )
    return bridge_trip_post_activity_df